# 01 — System architecture for customer support agents

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/01_customer_support_agent/01_architecture.ipynb)

**Prerequisites**:
- [00_first_principles.ipynb](./00_first_principles.ipynb) — intent classification, confidence calibration, multi-turn state

**What you will learn**:
- How a production support agent decomposes into five pipeline stages
- How to build a chunked knowledge base with embedding-based retrieval
- How to route intents using embedding centroids and cosine similarity
- How multi-stage retrieval improves precision over brute-force search
- How to design system prompts that enforce citation and tone guidelines
- How conversation state accumulates entities across turns

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "sentence-transformers>=2.2.0" "numpy>=1.24.0"

import numpy as np
from dataclasses import dataclass, field
from typing import Optional
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded ✓")

## Section 1 — System architecture overview

A production customer support agent is not a single LLM call — it is a **five-stage pipeline** where each stage has a clear contract:

```
┌──────────────┐     ┌───────────────┐     ┌──────────────────┐
│ User Message  │────▶│ Intent Router │────▶│ Knowledge RAG    │
└──────────────┘     └───────────────┘     └──────────────────┘
                                                    │
                                                    ▼
                      ┌───────────────┐     ┌──────────────────┐
                      │   Escalate    │◀────│ Confidence Gate  │
                      │  (to human)   │     └──────────────────┘
                      └───────────────┘             │ pass
                                                    ▼
                                            ┌──────────────────┐
                                            │ Response to User │
                                            └──────────────────┘
```

| Stage | Input | Output | Failure mode |
|-------|-------|--------|-------------|
| **Intent Router** | Raw user message | Intent label + confidence | Misroutes → wrong knowledge base |
| **Knowledge RAG** | Intent + query | Top-k document chunks | Irrelevant retrieval → hallucination |
| **Response Generator** | Query + context chunks | Draft response | Off-brand tone, missing citations |
| **Confidence Gate** | Response + retrieval scores | Pass / escalate decision | False confidence → wrong answer delivered |

Each component can be tested, measured, and improved independently.

In [ ]:
# Model the pipeline stages as explicit data structures

from dataclasses import dataclass, field
from typing import Optional


@dataclass
class PipelineStage:
    """Represents one stage of the support agent pipeline."""
    name: str
    input_type: str
    output_type: str
    description: str


@dataclass
class PipelineResult:
    """Captures the output of running a message through the full pipeline."""
    user_message: str
    intent: Optional[str] = None
    intent_confidence: float = 0.0
    retrieved_chunks: list[str] = field(default_factory=list)
    retrieval_scores: list[float] = field(default_factory=list)
    response: Optional[str] = None
    escalated: bool = False


PIPELINE = [
    PipelineStage("Intent Router", "str (user message)", "str (intent label), float (confidence)",
                  "Classifies the user's intent via embedding similarity to labeled centroids."),
    PipelineStage("Knowledge RAG", "str (intent), str (query)", "list[str] (chunks), list[float] (scores)",
                  "Retrieves relevant knowledge chunks filtered by intent category."),
    PipelineStage("Response Generator", "str (query), list[str] (context)", "str (draft response)",
                  "Generates a grounded response using retrieved context."),
    PipelineStage("Confidence Gate", "str (response), list[float] (scores)", "bool (escalate)",
                  "Decides whether the response is safe to send or needs human review."),
]

print("Agent Pipeline Stages")
print("=" * 70)
for i, stage in enumerate(PIPELINE, 1):
    print(f"\nStage {i}: {stage.name}")
    print(f"  Input:  {stage.input_type}")
    print(f"  Output: {stage.output_type}")
    print(f"  Role:   {stage.description}")

## Section 2 — Knowledge base design

The knowledge base is the foundation of a support agent. We need documents covering every topic the agent might encounter. For an e-commerce company, this includes:

- **Product documentation** — specs, features, compatibility
- **FAQ entries** — common questions and answers
- **Shipping policies** — delivery times, carriers, international
- **Return & refund policies** — eligibility, process, timelines

Each document gets chunked, embedded, and tagged with a category so the retriever can filter before searching.

In [ ]:
# Build a synthetic e-commerce knowledge base with ~20 documents

KNOWLEDGE_BASE: list[dict[str, str]] = [
    # Product docs
    {"category": "product_info", "title": "Wireless Headphones Pro",
     "content": "The Wireless Headphones Pro feature 40-hour battery life, active noise cancellation, "
                "Bluetooth 5.3, and USB-C fast charging. Compatible with iOS, Android, and Windows. "
                "Available in black, white, and midnight blue. Weight: 250g. Driver size: 40mm."},
    {"category": "product_info", "title": "Portable Speaker Max",
     "content": "The Portable Speaker Max delivers 360-degree sound with 20W output. IPX7 waterproof "
                "rating, 12-hour battery, and built-in microphone for calls. Pairs with up to 3 devices "
                "simultaneously. Dimensions: 180mm x 75mm. Weight: 680g."},
    {"category": "product_info", "title": "USB-C Fast Charger",
     "content": "65W GaN USB-C charger with dual ports (USB-C + USB-A). Supports PD 3.0 and QC 4.0. "
                "Compatible with laptops, tablets, and phones. Foldable prongs for travel. "
                "Includes 1m USB-C cable. Safety certified: UL, FCC, CE."},
    {"category": "product_info", "title": "Mechanical Keyboard Elite",
     "content": "Hot-swappable mechanical keyboard with RGB backlighting. Cherry MX Brown switches, "
                "N-key rollover, USB-C detachable cable. Aluminum frame. Includes wrist rest and "
                "keycap puller. Compatible with Windows and macOS."},
    {"category": "product_info", "title": "Wireless Mouse Ergo",
     "content": "Ergonomic wireless mouse with 4000 DPI optical sensor. Bluetooth and 2.4GHz dual "
                "connectivity. 6 programmable buttons, silent clicks. Battery lasts 90 days on single "
                "AA. Compatible with Windows, macOS, and Linux."},
    # FAQ
    {"category": "general", "title": "FAQ: Account creation",
     "content": "To create an account, visit our website and click 'Sign Up'. Enter your email, "
                "create a password (minimum 8 characters), and verify your email. You can also sign "
                "up with Google or Apple ID for faster registration."},
    {"category": "general", "title": "FAQ: Password reset",
     "content": "To reset your password, click 'Forgot Password' on the login page. Enter your "
                "registered email address. You will receive a reset link valid for 24 hours. "
                "If you don't receive the email, check your spam folder."},
    {"category": "general", "title": "FAQ: Order tracking",
     "content": "After placing an order, you will receive a confirmation email with a tracking number "
                "within 24 hours. Use the tracking number on our website under 'Track Order' or "
                "directly on the carrier's website (UPS, FedEx, or USPS)."},
    {"category": "general", "title": "FAQ: Warranty coverage",
     "content": "All products come with a 1-year limited warranty covering manufacturing defects. "
                "Warranty does not cover physical damage, water damage (except IPX-rated products), "
                "or unauthorized modifications. Contact support to file a warranty claim."},
    {"category": "general", "title": "FAQ: International orders",
     "content": "We ship to 45+ countries. International orders may be subject to customs duties "
                "and import taxes, which are the buyer's responsibility. Delivery takes 7-14 "
                "business days depending on destination."},
    # Shipping policies
    {"category": "order_status", "title": "Shipping: Standard delivery",
     "content": "Standard shipping takes 3-5 business days within the continental US. Orders placed "
                "before 2 PM EST ship same day. Free shipping on orders over $50. Tracking number "
                "provided via email within 24 hours of shipment."},
    {"category": "order_status", "title": "Shipping: Express delivery",
     "content": "Express shipping delivers in 1-2 business days. Available for $12.99 or free on "
                "orders over $100. Express orders placed before 12 PM EST ship same day. "
                "Not available for PO boxes or APO/FPO addresses."},
    {"category": "order_status", "title": "Shipping: Delays and issues",
     "content": "If your order is delayed beyond the estimated delivery date, please allow 2 extra "
                "business days before contacting support. Common causes: weather events, carrier "
                "volume surges, address verification holds. We will reship or refund if lost."},
    # Return / refund policies
    {"category": "billing", "title": "Returns: Standard return policy",
     "content": "Items can be returned within 30 days of delivery for a full refund. Products must "
                "be in original packaging, unused, with all accessories. Return shipping is free "
                "using our prepaid label. Refunds processed within 5-7 business days."},
    {"category": "billing", "title": "Returns: Defective items",
     "content": "Defective items can be returned within 90 days for a full refund or replacement. "
                "No original packaging required. We cover return shipping. Please include a brief "
                "description of the defect. Replacements ship within 2 business days."},
    {"category": "billing", "title": "Billing: Duplicate charges",
     "content": "If you see a duplicate charge, please wait 24 hours as pending authorizations "
                "often drop automatically. If the charge persists, contact support with your order "
                "number and we will issue a refund within 3 business days."},
    {"category": "billing", "title": "Billing: Refund timeline",
     "content": "Refunds are processed within 5-7 business days after we receive the returned item. "
                "Credit card refunds may take an additional 3-5 business days to appear on your "
                "statement. PayPal and store credit refunds are immediate upon processing."},
    # Technical support
    {"category": "technical_support", "title": "Tech: Bluetooth pairing issues",
     "content": "If your device won't pair: 1) Ensure Bluetooth is enabled on your phone/computer. "
                "2) Put the device in pairing mode (hold power button 5 seconds until LED flashes blue). "
                "3) Select the device from your Bluetooth settings. 4) If it fails, forget the device "
                "and try again. Factory reset: hold power + volume down for 10 seconds."},
    {"category": "technical_support", "title": "Tech: Firmware updates",
     "content": "Download our companion app (iOS/Android) to check for firmware updates. Connect your "
                "device via Bluetooth, open the app, and navigate to Settings > Firmware Update. "
                "Keep the device charged above 50% during updates. Updates typically take 3-5 minutes."},
    {"category": "technical_support", "title": "Tech: Battery not charging",
     "content": "If your device won't charge: 1) Try a different USB-C cable. 2) Clean the charging "
                "port with compressed air. 3) Ensure the charger outputs at least 5V/1A. 4) Try a "
                "different power outlet. If none of these work, the battery may need replacement — "
                "contact support for warranty service."},
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
print(f"Categories: {sorted(set(d['category'] for d in KNOWLEDGE_BASE))}")
for cat in sorted(set(d["category"] for d in KNOWLEDGE_BASE)):
    count = sum(1 for d in KNOWLEDGE_BASE if d["category"] == cat)
    print(f"  {cat}: {count} docs")

In [ ]:
# Chunk documents and build a simple embedding-based vector store

def chunk_document(doc: dict[str, str], max_chars: int = 300) -> list[dict]:
    """Split a document into overlapping chunks with metadata."""
    text = doc["content"]
    sentences = text.replace(". ", ".|").split("|")
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append({
                "text": current_chunk.strip(),
                "category": doc["category"],
                "title": doc["title"],
            })
            # Overlap: keep last sentence
            current_chunk = sentence
        else:
            current_chunk += " " + sentence if current_chunk else sentence
    if current_chunk.strip():
        chunks.append({
            "text": current_chunk.strip(),
            "category": doc["category"],
            "title": doc["title"],
        })
    return chunks


# Chunk all documents
all_chunks: list[dict] = []
for doc in KNOWLEDGE_BASE:
    all_chunks.extend(chunk_document(doc))

# Embed all chunks
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = model.encode(chunk_texts, show_progress_bar=True)

print(f"Total chunks: {len(all_chunks)}")
print(f"Embedding shape: {chunk_embeddings.shape}")
print(f"\nChunk length stats:")
lengths = [len(c["text"]) for c in all_chunks]
print(f"  Min: {min(lengths)} chars")
print(f"  Max: {max(lengths)} chars")
print(f"  Mean: {np.mean(lengths):.0f} chars")
print(f"\nSample chunk: {all_chunks[0]['title']}")
print(f"  '{all_chunks[0]['text'][:120]}...'")

## Section 3 — Intent routing with embeddings

Intent routing is the first pipeline stage. Rather than training a classifier, we use a **centroid-based approach**:

1. Define intent categories with 3–5 example phrases each
2. Encode examples → compute the centroid (mean embedding) per category
3. For a new query, compute cosine similarity to each centroid
4. Assign the category with highest similarity (if above threshold)

This is a zero-shot classifier that can be updated instantly by adding examples — no retraining needed.

In [ ]:
# Build an intent classifier using embedding centroids

from sklearn.metrics.pairwise import cosine_similarity as cos_sim

INTENT_EXAMPLES: dict[str, list[str]] = {
    "order_status": [
        "Where is my order?",
        "My package hasn't arrived yet",
        "Can I track my shipment?",
        "When will my order be delivered?",
        "I need an update on my delivery",
    ],
    "product_info": [
        "What are the specs of the headphones?",
        "Is this product waterproof?",
        "What colors does the speaker come in?",
        "How long does the battery last?",
        "Is it compatible with my Mac?",
    ],
    "billing": [
        "I was charged twice",
        "Where is my refund?",
        "I want to return this item",
        "How do I get my money back?",
        "The charge on my card is wrong",
    ],
    "technical_support": [
        "My headphones won't connect to Bluetooth",
        "The device isn't charging",
        "How do I update the firmware?",
        "The sound quality is distorted",
        "It keeps disconnecting from my phone",
    ],
    "complaint": [
        "This product is terrible",
        "I'm very unhappy with my purchase",
        "Worst customer service I've ever had",
        "I want to speak to a manager",
        "This is unacceptable, I demand a resolution",
    ],
    "general": [
        "How do I create an account?",
        "What's your warranty policy?",
        "Do you ship internationally?",
        "What are your business hours?",
        "How do I reset my password?",
    ],
}

# Compute centroid for each intent
intent_centroids: dict[str, np.ndarray] = {}
for intent, examples in INTENT_EXAMPLES.items():
    embeddings = model.encode(examples)
    intent_centroids[intent] = np.mean(embeddings, axis=0)

print(f"Intent centroids computed for {len(intent_centroids)} categories")
for intent in intent_centroids:
    print(f"  • {intent} ({len(INTENT_EXAMPLES[intent])} examples)")

In [ ]:
# Classify test queries and print results

def classify_intent(query: str, centroids: dict[str, np.ndarray],
                    threshold: float = 0.25) -> tuple[str, float]:
    """Classify a query's intent using cosine similarity to centroids."""
    query_emb = model.encode([query])
    best_intent = "unknown"
    best_score = -1.0
    for intent, centroid in centroids.items():
        score = cos_sim(query_emb, centroid.reshape(1, -1))[0][0]
        if score > best_score:
            best_score = score
            best_intent = intent
    if best_score < threshold:
        return "unknown", best_score
    return best_intent, float(best_score)


test_queries = [
    "Where's my package? It's been 5 days",
    "Does the speaker work underwater?",
    "I got charged $50 twice on my credit card",
    "My keyboard's RGB lights stopped working",
    "Your product literally fell apart after one week",
    "Can I use the charger for my laptop?",
    "I need to change my email address on my account",
    "What's the return window for opened items?",
    "The left earbud has no sound at all",
    "I want to cancel my order before it ships",
    "Do you ship to Canada?",
    "How do I pair my speaker with two phones?",
]

print(f"{'Query':<50} {'Intent':<20} {'Score':<6}")
print("=" * 76)
for q in test_queries:
    intent, score = classify_intent(q, intent_centroids)
    print(f"{q:<50} {intent:<20} {score:.3f}")

## Section 4 — Multi-stage retrieval

Brute-force semantic search over all chunks works but has two problems:
1. **Noise**: unrelated categories contribute false positives
2. **Speed**: searching all chunks is O(n) per query

A **two-stage retriever** solves both:
- **Stage 1**: Filter chunks by the detected intent category
- **Stage 2**: Semantic search only within the filtered set

This is analogous to how a human agent first thinks "this is a billing question" and then checks the billing knowledge base — not the entire company wiki.

In [ ]:
# Compare single-stage vs multi-stage retrieval

def retrieve_single_stage(query: str, top_k: int = 3) -> list[tuple[str, float, str]]:
    """Search all chunks regardless of category."""
    query_emb = model.encode([query])
    scores = cos_sim(query_emb, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(all_chunks[i]["text"], float(scores[i]), all_chunks[i]["category"])
            for i in top_indices]


def retrieve_multi_stage(query: str, intent: str, top_k: int = 3) -> list[tuple[str, float, str]]:
    """Filter by intent category first, then semantic search."""
    # Stage 1: filter by category
    filtered_indices = [i for i, c in enumerate(all_chunks) if c["category"] == intent]
    if not filtered_indices:
        return retrieve_single_stage(query, top_k)  # fallback
    # Stage 2: semantic search within filtered set
    filtered_embeddings = chunk_embeddings[filtered_indices]
    query_emb = model.encode([query])
    scores = cos_sim(query_emb, filtered_embeddings)[0]
    top_local = np.argsort(scores)[::-1][:top_k]
    return [(all_chunks[filtered_indices[i]]["text"], float(scores[i]),
             all_chunks[filtered_indices[i]]["category"]) for i in top_local]


# Test on queries where the distinction matters
test_cases = [
    ("I was charged twice for order #5521", "billing"),
    ("How long does the headphone battery last?", "product_info"),
    ("My Bluetooth keeps disconnecting", "technical_support"),
    ("When will my package arrive?", "order_status"),
]

for query, expected_cat in test_cases:
    intent, _ = classify_intent(query, intent_centroids)
    single = retrieve_single_stage(query, top_k=3)
    multi = retrieve_multi_stage(query, intent, top_k=3)
    single_relevant = sum(1 for _, _, cat in single if cat == expected_cat)
    multi_relevant = sum(1 for _, _, cat in multi if cat == expected_cat)
    print(f"Query: {query}")
    print(f"  Detected intent: {intent}")
    print(f"  Single-stage: {single_relevant}/3 relevant | top score={single[0][1]:.3f}")
    print(f"  Multi-stage:  {multi_relevant}/3 relevant | top score={multi[0][1]:.3f}")
    print(f"  Top result: '{multi[0][0][:80]}...'")
    print()

## Section 5 — Response generation prompt design

The response generator is the most visible component — it produces the text the customer actually reads. A well-designed system prompt enforces:

- **Role**: who the agent is and what brand it represents
- **Context injection**: where retrieved documents are placed
- **Citation requirements**: responses must reference source documents
- **Tone guidelines**: professional, empathetic, concise
- **Guardrails**: what the agent must never do (e.g., make promises, share internal data)

In [ ]:
# Design the system prompt template for response generation

SYSTEM_PROMPT_TEMPLATE = """You are a customer support agent for TechGear, an online electronics retailer.

ROLE:
- You are friendly, professional, and concise
- You help customers with orders, products, billing, and technical issues
- You NEVER make promises about specific outcomes (refunds, replacements) without policy backing

CONTEXT (retrieved from knowledge base):
{context}

INSTRUCTIONS:
1. Answer the customer's question using ONLY the information in CONTEXT above
2. If the context does not contain enough information, say "I don't have enough information to answer that fully. Let me connect you with a specialist."
3. Cite the source document when making factual claims, e.g. [Source: Shipping Policy]
4. Keep responses under 150 words
5. Use a warm, empathetic tone — acknowledge the customer's situation before providing solutions
6. If the customer is upset, lead with empathy before facts
7. Never reveal internal processes, ticket numbers, or system details
8. For billing disputes involving >$100, recommend escalation to a billing specialist

CONVERSATION HISTORY:
{history}

CUSTOMER MESSAGE:
{user_message}
"""


def build_prompt(user_message: str, context_chunks: list[tuple[str, float, str]],
                 history: list[dict[str, str]] | None = None) -> str:
    """Assemble the full prompt from template, context, and history."""
    context_block = "\n\n".join(
        f"[Document: {i+1}] (score: {score:.3f})\n{text}"
        for i, (text, score, _) in enumerate(context_chunks)
    )
    history_block = "\n".join(
        f"{turn['role'].upper()}: {turn['message']}"
        for turn in (history or [])
    ) or "(No prior conversation)"

    return SYSTEM_PROMPT_TEMPLATE.format(
        context=context_block,
        history=history_block,
        user_message=user_message,
    )


# Demo: build a prompt for a sample query
sample_query = "My headphones won't connect to my phone via Bluetooth"
intent, _ = classify_intent(sample_query, intent_centroids)
chunks = retrieve_multi_stage(sample_query, intent, top_k=2)
prompt = build_prompt(sample_query, chunks)

print("Generated prompt:")
print("=" * 70)
print(prompt)
print("=" * 70)
print(f"\nPrompt length: {len(prompt)} chars / ~{len(prompt)//4} tokens (approx)")

## Section 6 — Conversation state management

Real support conversations are multi-turn. The agent must track:
- **Message history**: full conversation for context
- **Extracted entities**: order IDs, product names, dates, amounts
- **Current intent**: may shift across turns (e.g., order inquiry → complaint)
- **Escalation score**: accumulates when the conversation suggests complexity

This state object travels through every pipeline stage and informs all decisions.

In [ ]:
# Build a ConversationState class with entity tracking and intent history

import re
from dataclasses import dataclass, field


@dataclass
class ConversationState:
    """Tracks all state for a multi-turn support conversation."""
    session_id: str
    history: list[dict[str, str]] = field(default_factory=list)
    entities: dict[str, str] = field(default_factory=dict)
    intent_history: list[str] = field(default_factory=list)
    escalation_score: float = 0.0
    turn_count: int = 0

    @property
    def current_intent(self) -> str | None:
        """Return the most recent intent classification."""
        return self.intent_history[-1] if self.intent_history else None

    def add_message(self, role: str, content: str) -> None:
        """Append a message to the conversation history."""
        self.history.append({"role": role, "message": content})
        if role == "user":
            self.turn_count += 1
            self._extract_entities(content)
            self._update_escalation(content)

    def _extract_entities(self, message: str) -> None:
        """Extract structured entities from a user message."""
        order_match = re.search(r"#?(\d{4,})", message)
        if order_match:
            self.entities["order_id"] = order_match.group(1)
        products = ["headphones", "speaker", "charger", "keyboard", "mouse"]
        for p in products:
            if p in message.lower():
                self.entities["product"] = p
        amount_match = re.search(r"\$([\d,.]+)", message)
        if amount_match:
            self.entities["amount"] = amount_match.group(1)
        email_match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", message)
        if email_match:
            self.entities["email"] = email_match.group(0)

    def _update_escalation(self, message: str) -> None:
        """Adjust escalation score based on message signals."""
        escalation_signals = [
            ("manager", 0.3), ("supervisor", 0.3), ("unacceptable", 0.2),
            ("lawsuit", 0.4), ("attorney", 0.4), ("terrible", 0.15),
            ("furious", 0.2), ("worst", 0.15), ("demand", 0.1),
        ]
        msg_lower = message.lower()
        for keyword, weight in escalation_signals:
            if keyword in msg_lower:
                self.escalation_score = min(1.0, self.escalation_score + weight)

    def record_intent(self, intent: str) -> None:
        """Record the classified intent for this turn."""
        self.intent_history.append(intent)

    def should_escalate(self, threshold: float = 0.5) -> bool:
        """Check if the conversation should be escalated to a human."""
        return self.escalation_score >= threshold

    def summary(self) -> str:
        """Return a compact summary of the current state."""
        return (
            f"Session {self.session_id} | Turn {self.turn_count} | "
            f"Intent: {self.current_intent} | "
            f"Entities: {self.entities} | "
            f"Escalation: {self.escalation_score:.2f}"
        )


# Simulate a 3-turn conversation
state = ConversationState(session_id="sess-001")

turns = [
    "Hi, I ordered the wireless headphones last week and they haven't arrived. Order #78432.",
    "I paid $89.99 and I'm getting really frustrated. It's been 8 days now.",
    "This is unacceptable. I want to speak to a manager about this.",
]

print("=== Conversation State Evolution ===\n")
for i, msg in enumerate(turns, 1):
    state.add_message("user", msg)
    intent, conf = classify_intent(msg, intent_centroids)
    state.record_intent(intent)
    print(f"Turn {i}: \"{msg}\"")
    print(f"  {state.summary()}")
    print(f"  Should escalate: {state.should_escalate()}")
    print()

print("Final state:")
print(f"  Entities: {state.entities}")
print(f"  Intent trajectory: {state.intent_history}")
print(f"  Escalation score: {state.escalation_score:.2f}")

## Exercises

1. **Add a new intent category**: Add `"account_management"` to the intent classifier with 5 example phrases (password changes, email updates, subscription management). Test it on 5 queries and verify it doesn't degrade existing category accuracy.

2. **Improve chunking strategy**: The current chunker splits on character count. Implement a **semantic chunker** that keeps related sentences together using embedding similarity between consecutive sentences. Compare retrieval quality on 3 test queries.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | A support agent is a five-stage pipeline: intent routing → knowledge retrieval → response generation → confidence gate → escalation |
| 2 | A tagged, chunked knowledge base with embeddings enables filtered semantic search |
| 3 | Centroid-based intent classification is zero-shot and instantly updatable — no retraining needed |
| 4 | Multi-stage retrieval (filter by intent, then search) outperforms brute-force search on precision |
| 5 | System prompts must enforce citations, tone, and guardrails to produce safe, on-brand responses |
| 6 | Conversation state management (entities, intent history, escalation score) is essential for multi-turn support |

## What's Next

- **Next**: [02_build.ipynb](./02_build.ipynb) — Build the full support agent with OpenAI integration, tool use, and confidence-based escalation
- **Related**: [03_evaluate.ipynb](./03_evaluate.ipynb) — Evaluate the agent with LLM-as-judge, hallucination detection, and cost analysis

## References & Further Reading

1. [Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks," 2020](https://arxiv.org/abs/2005.11401) — The foundational RAG paper that underpins the knowledge retrieval stage
2. [Reimers & Gurevych, "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks," 2019](https://arxiv.org/abs/1908.10084) — The embedding model architecture used for intent classification and retrieval
3. [Khattab & Zaharia, "ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction," 2020](https://arxiv.org/abs/2004.12832) — Advanced retrieval techniques for production systems